In [1]:
import pandas as pd
import re
import numpy as np

In [2]:
# Path to the folder where you downloaded the files
folder_path = '/itf-fi-ml/shared/users/ziyuzh/svm/data/opentarget/association_overall_direct'

# Read all parquet parts at once
df = pd.read_parquet(folder_path, engine='pyarrow')

# Show the first few rows
print(df.head(3))

      diseaseId         targetId     score  evidenceCount
0  DOID_0050890  ENSG00000001084  0.031799              4
1  DOID_0050890  ENSG00000004142  0.002217              1
2  DOID_0050890  ENSG00000004478  0.002217              1


In [3]:
df.columns

Index(['diseaseId', 'targetId', 'score', 'evidenceCount'], dtype='object')

In [4]:
all_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/timecut/disgent_with_time.csv')


In [8]:
import os
file_names = os.listdir('/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_lf_bag_cv')
selected_diseases = [item.split('.')[0] for item in file_names]
len(selected_diseases),selected_diseases[0]

(49, 'ICD10_D66')

In [19]:
selected_diseases

['ICD10_D66',
 'ICD10_C16',
 'ICD10_F31',
 'ICD10_I10',
 'ICD10_D83',
 'ICD10_L20',
 'ICD10_C23',
 'ICD10_N17',
 'ICD10_M34',
 'ICD10_I25',
 'ICD10_J45',
 'ICD10_I26',
 'ICD10_K44',
 'ICD10_C50',
 'ICD10_G30',
 'ICD10_C43',
 'ICD10_F90',
 'ICD10_N04',
 'ICD10_N46',
 'ICD10_D57',
 'ICD10_I50',
 'ICD10_C18',
 'ICD10_G40',
 'ICD10_J62',
 'ICD10_C53',
 'ICD10_I42',
 'ICD10_F72',
 'ICD10_G20',
 'ICD10_G10',
 'all_disease',
 'ICD10_J80',
 'ICD10_E11',
 'ICD10_L40',
 'ICD10_N80',
 'ICD10_G91',
 'ICD10_K51',
 'ICD10_M41',
 'ICD10_N97',
 'ICD10_F20',
 'ICD10_F01',
 'ICD10_M32',
 'ICD10_E66',
 'ICD10_G24',
 'ICD10_N18',
 'ICD10_C81',
 'ICD10_I63',
 'ICD10_G43',
 'ICD10_I70',
 'ICD10_C67']

In [9]:
icd2do_map = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/opentarget/ICD2DO.tsv',sep='\t')
icd2do_map.head(3)

,id,label,xrefs
0,DOID:0014667,disease of metabolism,ICD10CM:E88.9
1,DOID:0040008,isoniazide allergy,ICD10CM:Z88.1
2,DOID:0040010,mepivacaine allergy,ICD10CM:Z88.4


In [10]:
icd2do_map['icd'] = icd2do_map['xrefs'].str.split(':').str[1].str.split('.').str[0]

In [11]:
query_disease_map = icd2do_map[icd2do_map['icd'].isin([items[-3: ]for items in selected_diseases])]

In [12]:
query_disease_map = query_disease_map.copy()
query_disease_map['diseaseId'] = query_disease_map['id'].str.replace(':', '_', regex=False)

In [13]:
query_disease_map = query_disease_map.drop(columns=['id'])

In [14]:
ot_disease = pd.read_parquet('/itf-fi-ml/shared/users/ziyuzh/svm/data/opentarget/disease/disease.parquet')
ot_disease.head(2)

,id,code,name,description,dbXRefs,parents,synonyms,obsoleteTerms,obsoleteXRefs,children,ancestors,therapeuticAreas,descendants,ontology
0,DOID_0050890,http://purl.obolibrary.org/obo/DOID_0050890,synucleinopathy,A neurodegenerative disease that is characteri...,"[MESH:D000080874, MONDO:0000510, UMLS:C5191670...","[MONDO_0019052, MONDO_0021179, MONDO_0024237]",{'hasExactSynonym': ['alpha Synucleinopathies'...,[],[],"[EFO_0006792, EFO_1001050]","[MONDO_0024237, EFO_0005772, EFO_0000618, MOND...","[EFO_0000618, OTAR_0000018, OTAR_0000020]","[MONDO_0000211, MONDO_0016418, MONDO_0014889, ...","{'isTherapeuticArea': False, 'leaf': False, 's..."
1,DOID_10113,http://purl.obolibrary.org/obo/DOID_10113,trypanosomiasis,Infection with protozoa of the genus trypanosoma.,"[UMLS:C0041227, MONDO:0000940, ICD10CM:B56, Me...",[MONDO_0002428],{'hasExactSynonym': ['Trypanosoma caused disea...,[],[],"[MONDO_0001444, EFO_0005225, EFO_0008559]","[MONDO_0002428, EFO_0001067, EFO_0005741]",[EFO_0005741],"[EFO_0005225, EFO_0005529, EFO_0008559, MONDO_...","{'isTherapeuticArea': False, 'leaf': False, 's..."


In [15]:
query_disease_map = query_disease_map.rename(columns={'label': 'name'})
query_disease_direct_name_map = pd.merge(query_disease_map, ot_disease, on='name', how='inner')

In [16]:
########## directly mapped disease
query_disease_direct_name_map['icd'].unique().shape

(37,)

In [17]:
missing_disease = list(set([items[-3: ]for items in selected_diseases]) - set(query_disease_direct_name_map['icd'].unique().tolist()))

In [18]:
set(missing_disease) - set(icd2do_map[icd2do_map['icd'].isin(missing_disease)]['icd'].unique().tolist())
######## note: we should delete F90

{'F90', 'ase'}

In [20]:
near_disease_map = {
    'G20': ['Parkinsonism', 'Parkinson disease', 'late-onset Parkinson disease', 'young-onset Parkinson disease','hemiparkinsonism-hemiatrophy syndrome','Parkinson disease, dominant','Hereditary late-onset Parkinson disease'],
    'D57': ['sickle cell anemia',
            'sickle cell-beta-thalassemia disease syndrome',
            'sickle cell-hemoglobin c disease syndrome',
            'sickle cell-hemoglobin d disease syndrome',
            'sickle cell-hemoglobin E disease syndrome',
            'hereditary persistence of fetal hemoglobin-sickle cell disease syndrome',
            'sickle cell disease and related diseases',
            'Sickle cell - beta-thalassemia disease',
            'Sickle cell - hemoglobin C disease',
            'Sickle cell - hemoglobin D disease',
            'Sickle cell - hemoglobin E disease'],
    'G10': ['Huntington disease','juvenile Huntington disease'],
    'G30': ['Alzheimer disease',
            'Alzheimer disease type 1',
            'Alzheimer disease 3',
            'Alzheimer disease 18',
            'early-onset autosomal dominant Alzheimer disease',
            'familial Alzheimer disease',
            'late-onset Alzheimers disease'],
    'J80': ['adult acute respiratory distress syndrome','acute respiratory distress syndrome'],
    'L80': ['Vitiligo'],
    'N97': ['anovulation'],
    'L20': ['recalcitrant atopic dermatitis'],
    'C81': ['classic Hodgkin lymphoma',
            'Hodgkins lymphoma',
            'nodular sclerosis Hodgkin lymphoma',
            'Splenic Hodgkin Lymphoma',
            'Hodgkins lymphoma, mixed cellularity'],
    'C43': ['cutaneous melanoma',
            'amelanotic skin melanoma',
            'amelanotic melanoma',
            'superficial spreading melanoma',
            'lentigo maligna melanoma',
            'nodular melanoma',
            'desmoplastic melanoma',
            'spindle cell melanoma',
            'childhood malignant melanoma',
            'melanoma',
            'metastatic melanoma',
            'melanoma, cutaneous malignant, susceptibility to, 1',
            'melanoma, cutaneous malignant, susceptibility to, 2',
            'melanoma, cutaneous malignant, susceptibility to, 3',
            'melanoma, cutaneous malignant, susceptibility to, 8',
            'melanoma, cutaneous malignant, susceptibility to, 9',
            'susceptibility to familial cutaneous melanoma',
            'familial melanoma',
            'familial atypical multiple mole melanoma syndrome']
}

In [21]:
query_disease_missing = query_disease_map[query_disease_map['icd'].isin(missing_disease)]

In [22]:
# Make a copy to avoid SettingWithCopyWarning
query_disease_missing = query_disease_missing.copy()

# Expand rows
expanded = query_disease_missing.explode('icd').apply(
    lambda row: pd.DataFrame({
        **{col: [row[col]] * len(near_disease_map.get(row['icd'], [])) for col in query_disease_missing.columns},
        'name': near_disease_map.get(row['icd'], [])
    }),
    axis=1
)

# Concatenate the resulting small DataFrames
query_disease_expanded = pd.concat(expanded.tolist(), ignore_index=True)

In [23]:
query_disease_missing = pd.merge(query_disease_expanded, ot_disease, on='name', how='inner')
query_disease_missing['icd'].unique().shape

(9,)

In [24]:
query_disease_clean_map = pd.concat([query_disease_missing, query_disease_direct_name_map], ignore_index=True)

In [25]:
query_disease_clean_map['icd'].unique().shape

(46,)

In [26]:
query_dga = pd.merge(query_disease_clean_map[['id','icd']].rename(columns={'id': 'diseaseId'}), df, on='diseaseId', how='inner')

In [106]:
all_df.head(2)

,disease_id,omim,hpo,disease_name,gene_id,score,first_pub_year,last_pub_year,ei,dsi,dpi,uniprot_id,string_id,ori_annotation
0,ICD10_C16,OMIM_613659,HPO_HP:0012126,Malignant neoplasm of stomach,ERBB2,1.0,2004.0,2007.0,0.917,0.298,0.957,P04626,9606.ENSP00000269571,True
1,ICD10_C16,OMIM_613659,HPO_HP:0012126,Malignant neoplasm of stomach,PIK3CA,1.0,2004.0,2023.0,0.978,0.275,0.957,P42336,9606.ENSP00000263967,True


In [27]:
query_dga['disease_id'] = 'ICD10_'+query_dga['icd']

In [28]:
query_dga

,diseaseId,icd,targetId,score,evidenceCount,disease_id
0,HP_0001300,G20,ENSG00000002726,0.001478,1,ICD10_G20
1,HP_0001300,G20,ENSG00000003393,0.004435,2,ICD10_G20
2,HP_0001300,G20,ENSG00000003989,0.004435,1,ICD10_G20
3,HP_0001300,G20,ENSG00000004139,0.001848,2,ICD10_G20
4,HP_0001300,G20,ENSG00000004142,0.008203,4,ICD10_G20
...,...,...,...,...,...,...
386751,EFO_0001073,E66,ENSG00000284862,0.020376,1,ICD10_E66
386752,EFO_0001073,E66,ENSG00000285043,0.039944,3,ICD10_E66
386753,EFO_0001073,E66,ENSG00000286131,0.284720,2,ICD10_E66
386754,EFO_0001073,E66,ENSG00000310517,0.279751,4,ICD10_E66


In [35]:
len(query_dga['targetId'].unique().tolist())

22029

In [30]:
import mygene

In [36]:
mg = mygene.MyGeneInfo()
# Query mygene for UniProt and Entrez gene ID mappings
results = mg.querymany(
    query_dga['targetId'].unique().tolist(),
    scopes='ensembl.gene',
    fields='uniprot',
    species='human'
)
results_df = pd.DataFrame(results)

results_df['query'].unique().shape

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


5 input query terms found dup hits:	[('ENSG00000282304', 2), ('ENSG00000249738', 2), ('ENSG00000188660', 2), ('ENSG00000280018', 2), ('E
7 input query terms found no hit:	['ENSG00000168078', 'ENSG00000189144', 'ENSG00000291190', 'ENSG00000281376', 'ENSG00000310560', 'ENS


(22029,)

In [38]:
results_df['uniprot_ids'] = results_df['uniprot'].apply(
    lambda x: list(x.values())[0] if isinstance(x, dict) and 'Swiss-Prot' in x else None)

In [40]:
results_df = results_df[~results_df['uniprot_ids'].isna()]
len(results_df['query'].unique())

18002

In [41]:
results_df

,query,_id,_score,uniprot,notfound,uniprot_ids
0,ENSG00000002726,26,31.492868,"{'Swiss-Prot': 'P19801', 'TrEMBL': ['C9J0G8', ...",NaN,P19801
1,ENSG00000003393,57679,30.840534,"{'Swiss-Prot': 'Q96Q42', 'TrEMBL': ['A0A7P0T8F...",NaN,Q96Q42
2,ENSG00000003989,6542,31.492868,"{'Swiss-Prot': 'P52569', 'TrEMBL': ['A0A1W2PR0...",NaN,P52569
3,ENSG00000004139,23098,31.492868,"{'Swiss-Prot': 'Q6SZW1', 'TrEMBL': ['Q05B42', ...",NaN,Q6SZW1
4,ENSG00000004142,26073,32.483240,"{'Swiss-Prot': 'Q9Y2S7', 'TrEMBL': 'B4DEM9'}",NaN,Q9Y2S7
...,...,...,...,...,...,...
22027,ENSG00000185040,102723555,32.483240,"{'Swiss-Prot': 'A6NNV3', 'TrEMBL': ['A0A0J9YYB...",NaN,A6NNV3
22028,ENSG00000186844,353131,32.483402,{'Swiss-Prot': 'Q5T7P2'},NaN,Q5T7P2
22029,ENSG00000221887,284293,32.483240,"{'Swiss-Prot': ['P0C7T4', 'A8MTL9']}",NaN,"[P0C7T4, A8MTL9]"
22030,ENSG00000255855,100129053,31.492868,{'Swiss-Prot': 'A0A1W2PPD8'},NaN,A0A1W2PPD8


In [42]:
map_df = pd.DataFrame({
    'ensg': results_df['query'],
    'uniport': results_df['uniprot_ids']})

# Step 2: Explode the list of proteins to one per row
map_df = map_df.explode('uniport').reset_index(drop=True)

In [43]:
len(map_df)

18098

In [45]:
query_dga = query_dga.rename(columns={'targetId': 'ensg'})
mapped_ot_dga = pd.merge(query_dga, map_df, on='ensg', how='left')

In [46]:
mapped_ot_dga

,diseaseId,icd,ensg,score,evidenceCount,disease_id,uniport
0,HP_0001300,G20,ENSG00000002726,0.001478,1,ICD10_G20,P19801
1,HP_0001300,G20,ENSG00000003393,0.004435,2,ICD10_G20,Q96Q42
2,HP_0001300,G20,ENSG00000003989,0.004435,1,ICD10_G20,P52569
3,HP_0001300,G20,ENSG00000004139,0.001848,2,ICD10_G20,Q6SZW1
4,HP_0001300,G20,ENSG00000004142,0.008203,4,ICD10_G20,Q9Y2S7
...,...,...,...,...,...,...,...
388638,EFO_0001073,E66,ENSG00000284862,0.020376,1,ICD10_E66,Q9UFE4
388639,EFO_0001073,E66,ENSG00000285043,0.039944,3,ICD10_E66,NaN
388640,EFO_0001073,E66,ENSG00000286131,0.284720,2,ICD10_E66,NaN
388641,EFO_0001073,E66,ENSG00000310517,0.279751,4,ICD10_E66,P20810


In [47]:
mapped_ot_dga[['score', 'disease_id', 'uniport']].to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/opentarget/ot_dga_newest.csv',index=False)

In [19]:

import pandas as pd


In [8]:
# import json
# import pandas as pd
# import re
# import mygene

# # File path to the full 2019 JSON file
# input_file = "/itf-fi-ml/shared/users/ziyuzh/svm/data/opentarget/2019/19.02_association_data.json"

# rows = []

# with open(input_file, "r") as f:
#     for line in f:
#         obj = json.loads(line)
#         if obj['is_direct']:
#             gene_id = obj["target"]["id"]
#             score = obj['association_score']['overall']
#             disease_id = obj["disease"]["id"]
#             rows.append({
#                     "gene_id": gene_id,
#                     "EFO": disease_id,
#                     "score": score
#                 })
# df = pd.DataFrame(rows)

In [10]:
### all before 2025
ot_dgas_2025 = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/opentarget/ot_dga_newest.csv')
ot_dgas_2019 = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/opentarget/2019/row_dga_2019.csv')

In [13]:
ot_dgas_2019.rename(columns={'uniprot_ids':'uniport'},inplace=True)

In [12]:
ot_dgas_2025.head(3)

,score,disease_id,uniport
0,0.001478,ICD10_G20,P19801
1,0.004435,ICD10_G20,Q96Q42
2,0.004435,ICD10_G20,P52569


In [ ]:
df3 = ot_dgas_2025.merge(ot_dgas_2019[['disease_id','uniport']], 
                on=['disease_id','uniport'], 
                how='left', 
                indicator=True)

df3 = df3[df3['_merge'] == 'left_only'].drop(columns=['_merge'])


In [16]:
len(ot_dgas_2019),len(ot_dgas_2025),len(df3)

(687914, 388643, 250021)

In [17]:
df3.head(3)

,score,disease_id,uniport
1,0.004435,ICD10_G20,Q96Q42
2,0.004435,ICD10_G20,P52569
4,0.008203,ICD10_G20,Q9Y2S7


In [18]:
df3.to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/opentarget/ot_external_19_25.csv',index=False)

In [20]:
ot_dgas = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/opentarget/ot_external_19_25.csv')

ot_dgas = ot_dgas.sort_values('score', ascending=False) \
                 .drop_duplicates(subset=['disease_id','uniport'], keep='first')


In [22]:
ot_dgas.to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/opentarget/ot_external_19_25.csv',index=False)